# 🌐 Global Internet Speeds 2025 — Exploratory Data Analysis
**Author:** Yash Gupta | **Role:** Business Analyst | **Date:** April 2026

---
> **Dataset:** 157 countries • 2 metrics: Broadband & Mobile Download Speeds (Mbps)  
> **Objective:** Understand global internet connectivity disparities, identify regional patterns, and surface actionable business insights.

---
## 📋 Table of Contents
1. [Environment Setup](#1-environment-setup)
2. [Data Loading & First Look](#2-data-loading--first-look)
3. [Data Quality & Missing Values](#3-data-quality--missing-values)
4. [Descriptive Statistics](#4-descriptive-statistics)
5. [Top & Bottom Country Rankings](#5-top--bottom-country-rankings)
6. [Speed Distribution Analysis](#6-speed-distribution-analysis)
7. [Regional Analysis](#7-regional-analysis)
8. [Broadband vs Mobile Comparison](#8-broadband-vs-mobile-comparison)
9. [Correlation Analysis](#9-correlation-analysis)
10. [Key Business Insights & Recommendations](#10-key-business-insights--recommendations)


## 1. Environment Setup

In [ ]:
# Install required libraries (uncomment if needed)
# !pip install pandas matplotlib seaborn plotly

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Visual theme ──────────────────────────────────────────────────────────
PRIMARY, SECONDARY, ACCENT, DANGER = '#1E3A5F', '#2E86AB', '#F18F01', '#E63946'
BG = '#F7F9FC'
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False, 'axes.spines.right': False
})

print('✅ Libraries loaded successfully')
print(f'   pandas  {pd.__version__}')
print(f'   numpy   {np.__version__}')
print(f'   matplotlib {matplotlib.__version__}')
print(f'   seaborn {sns.__version__}')


## 2. Data Loading & First Look

In [ ]:
# Load dataset
df = pd.read_csv('global_internet_speeds_2025.csv')
df.columns = ['country', 'broadband', 'mobile']   # standardise column names

print(f'Shape: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Countries: {df["country"].nunique()}')
df.head(10)


In [ ]:
# Column data types and non-null counts
df.info()


In [ ]:
# Region mapping — assign each country to a world region
REGION = {
    # South Asia
    'Afghanistan':'South Asia','Bangladesh':'South Asia','India':'South Asia',
    'Nepal':'South Asia','Pakistan':'South Asia','Sri Lanka':'South Asia','Maldives':'South Asia',
    # East Asia
    'China':'East Asia','Hong Kong':'East Asia','Japan':'East Asia',
    'South Korea':'East Asia','Taiwan':'East Asia','Macau':'East Asia','Mongolia':'East Asia',
    # Southeast Asia
    'Brunei':'Southeast Asia','Cambodia':'Southeast Asia','Indonesia':'Southeast Asia',
    'Laos':'Southeast Asia','Malaysia':'Southeast Asia','Myanmar':'Southeast Asia',
    'Philippines':'Southeast Asia','Singapore':'Southeast Asia','Thailand':'Southeast Asia','Vietnam':'Southeast Asia',
    # Central Asia
    'Armenia':'Central Asia','Azerbaijan':'Central Asia','Georgia':'Central Asia',
    'Kazakhstan':'Central Asia','Kyrgyzstan':'Central Asia','Tajikistan':'Central Asia','Uzbekistan':'Central Asia',
    # Middle East
    'Bahrain':'Middle East','Iran':'Middle East','Iraq':'Middle East','Israel':'Middle East',
    'Jordan':'Middle East','Kuwait':'Middle East','Lebanon':'Middle East','Oman':'Middle East',
    'Palestine':'Middle East','Qatar':'Middle East','Saudi Arabia':'Middle East','Syria':'Middle East',
    'Turkey':'Middle East','United Arab Emirates':'Middle East','Yemen':'Middle East',
    # Europe
    'Albania':'Europe','Austria':'Europe','Belarus':'Europe','Belgium':'Europe',
    'Bosnia and Herzegovina':'Europe','Bulgaria':'Europe','Croatia':'Europe','Cyprus':'Europe',
    'Czech Republic':'Europe','Denmark':'Europe','Estonia':'Europe','Finland':'Europe',
    'France':'Europe','Germany':'Europe','Greece':'Europe','Hungary':'Europe','Iceland':'Europe',
    'Ireland':'Europe','Italy':'Europe','Kosovo':'Europe','Latvia':'Europe','Lithuania':'Europe',
    'Luxembourg':'Europe','Malta':'Europe','Moldova':'Europe','Montenegro':'Europe',
    'Netherlands':'Europe','North Macedonia':'Europe','Norway':'Europe','Poland':'Europe',
    'Portugal':'Europe','Romania':'Europe','Russia':'Europe','San Marino':'Europe',
    'Serbia':'Europe','Slovakia':'Europe','Slovenia':'Europe','Spain':'Europe','Sweden':'Europe',
    'Switzerland':'Europe','Ukraine':'Europe','United Kingdom':'Europe',
    # Africa
    'Algeria':'Africa','Angola':'Africa','Benin':'Africa','Botswana':'Africa',
    'Burkina Faso':'Africa','Cameroon':'Africa','Cape Verde':'Africa','DR Congo':'Africa',
    'Egypt':'Africa','Ethiopia':'Africa','Eswatini':'Africa','Gabon':'Africa','Ghana':'Africa',
    'Ivory Coast':'Africa','Kenya':'Africa','Lesotho':'Africa','Libya':'Africa',
    'Madagascar':'Africa','Mali':'Africa','Mauritania':'Africa','Mauritius':'Africa',
    'Morocco':'Africa','Mozambique':'Africa','Namibia':'Africa','Niger':'Africa',
    'Nigeria':'Africa','Rwanda':'Africa','Senegal':'Africa','Somalia':'Africa',
    'South Africa':'Africa','Tanzania':'Africa','Togo':'Africa','Tunisia':'Africa',
    'Uganda':'Africa','Zambia':'Africa','Zimbabwe':'Africa',
    # Americas
    'Canada':'North America','Cuba':'North America','Dominican Republic':'North America',
    'El Salvador':'North America','Guatemala':'North America','Haiti':'North America',
    'Honduras':'North America','Jamaica':'North America','Mexico':'North America',
    'Nicaragua':'North America','United States':'North America',
    'Antigua and Barbuda':'Caribbean','Bahamas':'Caribbean','Belize':'Caribbean',
    'Grenada':'Caribbean','Saint Kitts and Nevis':'Caribbean','Trinidad and Tobago':'Caribbean',
    'Argentina':'South America','Bolivia':'South America','Brazil':'South America',
    'Chile':'South America','Colombia':'South America','Costa Rica':'South America',
    'Ecuador':'South America','Panama':'South America','Paraguay':'South America',
    'Peru':'South America','Suriname':'South America','Uruguay':'South America','Venezuela':'South America',
    # Oceania
    'Australia':'Oceania','Fiji':'Oceania','New Zealand':'Oceania',
}
df['region'] = df['country'].map(REGION).fillna('Other')
print('Region distribution:')
print(df['region'].value_counts().to_string())


## 3. Data Quality & Missing Values
A key part of any BA project is understanding **what data you have vs. what you don't** — and why gaps exist.


In [ ]:
# Missing values summary
missing = df.isnull().sum()
pct     = (missing / len(df) * 100).round(1)
quality = pd.DataFrame({'Missing Count': missing, 'Missing %': pct,
                          'Available Count': len(df) - missing,
                          'Available %': (100 - pct).round(1)})
print('=== DATA QUALITY REPORT ===')
print(quality.drop('country').to_string())

# Countries with NO data at all
no_data = df[df['broadband'].isna() & df['mobile'].isna()]
print(f'\nCountries with BOTH metrics missing: {len(no_data)}')
if len(no_data): print(no_data['country'].tolist())

# Countries with only broadband
bb_only = df[df['broadband'].notna() & df['mobile'].isna()]
print(f'\nCountries with broadband only: {len(bb_only)}')

# Countries with both
both = df.dropna(subset=['broadband','mobile'])
print(f'Countries with BOTH metrics: {len(both)}')


In [ ]:
# Visualise data coverage
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i, (col, label) in enumerate([('broadband','Broadband'),('mobile','Mobile')]):
    avail   = df[col].notna().sum()
    miss    = df[col].isna().sum()
    ax      = axes[i]
    sizes   = [avail, miss]
    lbls    = [f'Available\n{avail}', f'Missing\n{miss}']
    colors  = [SECONDARY, '#E0E0E0']
    wedges, texts, autotexts = ax.pie(sizes, labels=lbls, colors=colors,
                                       autopct='%1.1f%%', startangle=90,
                                       wedgeprops={'edgecolor':'white','linewidth':2})
    autotexts[0].set_color('white'); autotexts[0].set_fontweight('bold')
    ax.set_title(f'{label} Coverage', fontsize=12, fontweight='bold', color=PRIMARY)
fig.suptitle('Data Coverage — 157 Countries (2025)', fontsize=13, fontweight='bold', color=PRIMARY)
plt.tight_layout()
plt.show()
print('\n⚠️  Note: 53 countries (33.8%) lack mobile speed data — primarily in Africa, the Caribbean, and smaller island nations.')
print('   Broadband data is near-complete with only 1 country (Eswatini) missing.')


## 4. Descriptive Statistics
Understanding the central tendency, spread, and outliers in speed data.


In [ ]:
df_bb  = df.dropna(subset=['broadband'])
df_mob = df.dropna(subset=['mobile'])

stats = pd.DataFrame({
    'Broadband': df_bb['broadband'].describe(),
    'Mobile':    df_mob['mobile'].describe()
}).round(2)
stats.index = ['Count','Mean','Std Dev','Min','25th %ile','Median','75th %ile','Max']
print('=== DESCRIPTIVE STATISTICS ===')
print(stats.to_string())

# IQR-based analysis
for col, label in [('broadband','Broadband'),('mobile','Mobile')]:
    d = df[col].dropna()
    q1, q3 = d.quantile(0.25), d.quantile(0.75)
    iqr     = q3 - q1
    outliers= d[(d < q1-1.5*iqr) | (d > q3+1.5*iqr)]
    print(f'\n{label} — IQR: {iqr:.1f} Mbps | Outliers (IQR method): {len(outliers)}')
    print(f'  Countries: {df.loc[outliers.index, "country"].tolist()}')


## 5. Top & Bottom Country Rankings
Rankings reveal the **digital divide** between leading and lagging nations.


In [ ]:
# ── Top 15 Broadband ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

top15_bb = df_bb.nlargest(15,'broadband').sort_values('broadband')
ax = axes[0]
colors   = [ACCENT if v>300 else SECONDARY for v in top15_bb['broadband']]
bars = ax.barh(top15_bb['country'], top15_bb['broadband'], color=colors, edgecolor='white', height=0.7)
for bar in bars:
    ax.text(bar.get_width()+4, bar.get_y()+bar.get_height()/2,
            f'{bar.get_width():.1f}', va='center', fontsize=9, fontweight='bold', color=PRIMARY)
ax.set_xlabel('Speed (Mbps)'); ax.set_title('Top 15 — Broadband', fontsize=12, fontweight='bold', color=PRIMARY)
ax.set_xlim(0, top15_bb['broadband'].max()*1.2)

# ── Bottom 15 Broadband ─────────────────────────────────────────────────
bot15_bb = df_bb.nsmallest(15,'broadband').sort_values('broadband', ascending=False)
ax = axes[1]
colors   = [DANGER if v<10 else '#FFA07A' for v in bot15_bb['broadband']]
bars = ax.barh(bot15_bb['country'], bot15_bb['broadband'], color=colors, edgecolor='white', height=0.7)
for bar in bars:
    ax.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
            f'{bar.get_width():.1f}', va='center', fontsize=9, fontweight='bold')
ax.set_xlabel('Speed (Mbps)'); ax.set_title('Bottom 15 — Broadband', fontsize=12, fontweight='bold', color=PRIMARY)
ax.set_xlim(0, bot15_bb['broadband'].max()*1.4)

plt.suptitle('Broadband Download Speed Rankings (2025)', fontsize=14, fontweight='bold', color=PRIMARY)
plt.tight_layout()
plt.show()


In [ ]:
# ── Top 15 Mobile ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

top15_mob = df_mob.nlargest(15,'mobile').sort_values('mobile')
ax = axes[0]
colors    = [ACCENT if v>300 else SECONDARY for v in top15_mob['mobile']]
bars = ax.barh(top15_mob['country'], top15_mob['mobile'], color=colors, edgecolor='white', height=0.7)
for bar in bars:
    ax.text(bar.get_width()+5, bar.get_y()+bar.get_height()/2,
            f'{bar.get_width():.1f}', va='center', fontsize=9, fontweight='bold', color=PRIMARY)
ax.set_xlabel('Speed (Mbps)'); ax.set_title('Top 15 — Mobile', fontsize=12, fontweight='bold', color=PRIMARY)
ax.set_xlim(0, top15_mob['mobile'].max()*1.18)

# Bottom 15 Mobile
bot15_mob = df_mob.nsmallest(15,'mobile').sort_values('mobile', ascending=False)
ax = axes[1]
colors    = [DANGER if v<20 else '#FFA07A' for v in bot15_mob['mobile']]
bars = ax.barh(bot15_mob['country'], bot15_mob['mobile'], color=colors, edgecolor='white', height=0.7)
for bar in bars:
    ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
            f'{bar.get_width():.1f}', va='center', fontsize=9, fontweight='bold')
ax.set_xlabel('Speed (Mbps)'); ax.set_title('Bottom 15 — Mobile', fontsize=12, fontweight='bold', color=PRIMARY)
ax.set_xlim(0, bot15_mob['mobile'].max()*1.4)

plt.suptitle('Mobile Download Speed Rankings (2025)', fontsize=14, fontweight='bold', color=PRIMARY)
plt.tight_layout()
plt.show()


In [ ]:
# Full ranked tables
print('=== TOP 10 BROADBAND ===')
print(df_bb.nlargest(10,'broadband')[['country','broadband','region']].reset_index(drop=True).to_string())
print('\n=== TOP 10 MOBILE ===')
print(df_mob.nlargest(10,'mobile')[['country','mobile','region']].reset_index(drop=True).to_string())


## 6. Speed Distribution Analysis
Histograms reveal how speeds are spread across countries — and the presence of **right-skewed distributions** typical of connectivity data.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, color, label in [
    (axes[0], 'broadband', PRIMARY, 'Broadband'),
    (axes[1], 'mobile',    ACCENT,  'Mobile')]:
    data = df[col].dropna()
    ax.hist(data, bins=25, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(data.mean(),   color=DANGER,    linestyle='--', lw=2, label=f'Mean: {data.mean():.1f}')
    ax.axvline(data.median(), color='#FF6B35', linestyle='-.',lw=2, label=f'Median: {data.median():.1f}')
    ax.set_xlabel('Speed (Mbps)', fontsize=11)
    ax.set_ylabel('Number of Countries', fontsize=11)
    ax.set_title(f'{label} Speed Distribution', fontsize=12, fontweight='bold', color=PRIMARY)
    ax.legend(fontsize=10)

plt.suptitle('Distribution of Internet Speeds — All Countries (2025)', fontsize=13, fontweight='bold', color=PRIMARY)
plt.tight_layout()
plt.show()

for col, label in [('broadband','Broadband'),('mobile','Mobile')]:
    d = df[col].dropna()
    skew = d.skew()
    kurt = d.kurt()
    print(f'{label}: Skewness={skew:.3f} | Kurtosis={kurt:.3f}')
    print(f'  → {"Right-skewed" if skew>0 else "Left-skewed"}: mean ({d.mean():.1f}) > median ({d.median():.1f}) = pulled by fast nations\n')


## 7. Regional Analysis
Breaking down speeds by **world region** highlights structural patterns in infrastructure investment, economic development, and policy.


In [ ]:
df_both = df.dropna(subset=['broadband','mobile'])
df_both_reg = df_both[df_both['region']!='Other']

reg_avg = df_both_reg.groupby('region')[['broadband','mobile']].mean().sort_values('broadband', ascending=False)
reg_cnt = df_both_reg.groupby('region')[['broadband','mobile']].count()

print('=== REGIONAL AVERAGES (countries with both metrics) ===')
summary = pd.concat([reg_avg.round(1), reg_cnt.rename(columns={'broadband':'bb_count','mobile':'mob_count'})], axis=1)
summary.columns = ['Avg Broadband (Mbps)','Avg Mobile (Mbps)','BB Count','Mob Count']
print(summary.to_string())


In [ ]:
# Regional boxplot — Broadband
df_bb_reg = df.dropna(subset=['broadband'])
df_bb_reg = df_bb_reg[df_bb_reg['region']!='Other']
region_order = df_bb_reg.groupby('region')['broadband'].median().sort_values(ascending=False).index.tolist()

fig, ax = plt.subplots(figsize=(13, 6))
sns.boxplot(data=df_bb_reg, x='broadband', y='region', order=region_order,
            palette='Set2', width=0.55, linewidth=1.2, fliersize=4, ax=ax)
ax.set_xlabel('Broadband Speed (Mbps)', fontsize=11)
ax.set_ylabel('Region', fontsize=11)
ax.set_title('Broadband Speed Distribution by Region (2025)', fontsize=13, fontweight='bold', color=PRIMARY)
plt.tight_layout()
plt.show()


In [ ]:
# Grouped bar — regional average broadband vs mobile
fig, ax = plt.subplots(figsize=(13, 6))
x  = np.arange(len(reg_avg))
w  = 0.38
b1 = ax.bar(x-w/2, reg_avg['broadband'], width=w, color=PRIMARY, label='Broadband', edgecolor='white')
b2 = ax.bar(x+w/2, reg_avg['mobile'],    width=w, color=ACCENT,   label='Mobile',    edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(reg_avg.index, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Avg Speed (Mbps)', fontsize=11)
ax.set_title('Average Speed by Region — Broadband vs Mobile (2025)', fontsize=13, fontweight='bold', color=PRIMARY)
ax.legend(fontsize=10)
for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
            f'{bar.get_height():.0f}', ha='center', fontsize=7.5)
plt.tight_layout()
plt.show()


## 8. Broadband vs Mobile Comparison
Some countries lead on **mobile** (often due to 5G investment), others rely more on **fixed broadband**. This gap reveals infrastructure strategy.


In [ ]:
# Scatter plot — Broadband vs Mobile per country
region_colors = {
    'Europe': SECONDARY, 'East Asia': ACCENT, 'North America': '#6A4C93',
    'South America': '#F72585', 'Southeast Asia': '#4CC9F0', 'Middle East': '#F77F00',
    'South Asia': '#80B918', 'Africa': '#E63946', 'Oceania': '#3A86FF',
    'Central Asia': '#8D99AE', 'Caribbean': '#06D6A0', 'Other': '#999'
}
fig, ax = plt.subplots(figsize=(11, 8))
for region, grp in df_both.groupby('region'):
    ax.scatter(grp['broadband'], grp['mobile'], label=region,
               color=region_colors.get(region,'#999'), alpha=0.85, s=65,
               edgecolors='white', linewidths=0.5)
# Annotate notable countries
notable = ['United Arab Emirates','Qatar','Singapore','Kuwait','France','United States']
for _, row in df_both[df_both['country'].isin(notable)].iterrows():
    ax.annotate(row['country'], (row['broadband'], row['mobile']),
                textcoords='offset points', xytext=(6, 3), fontsize=7.5, color='#333')
mx = max(df_both['broadband'].max(), df_both['mobile'].max()) + 30
ax.plot([0, mx], [0, mx], '--', color='#999', lw=1.5, label='Parity line (BB=Mobile)')
ax.set_xlabel('Broadband Speed (Mbps)', fontsize=11)
ax.set_ylabel('Mobile Speed (Mbps)', fontsize=11)
ax.set_title('Broadband vs Mobile Speed — Country Comparison (2025)', fontsize=13, fontweight='bold', color=PRIMARY)
ax.legend(bbox_to_anchor=(1.01,1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# Speed gap analysis (mobile - broadband)
df_both2 = df_both.copy()
df_both2['gap'] = df_both2['mobile'] - df_both2['broadband']

top_mob_dom = df_both2.nlargest(12,'gap')[['country','broadband','mobile','gap','region']]
top_bb_dom  = df_both2.nsmallest(12,'gap')[['country','broadband','mobile','gap','region']]

print('Countries where MOBILE is much FASTER than broadband (top 12):')
print(top_mob_dom.to_string(index=False))
print('\nCountries where BROADBAND is much FASTER than mobile (top 12):')
print(top_bb_dom.to_string(index=False))


In [ ]:
# Gap bar chart
combined = pd.concat([df_both2.nlargest(12,'gap'), df_both2.nsmallest(12,'gap')]).drop_duplicates()
combined_sorted = combined.sort_values('gap')

fig, ax = plt.subplots(figsize=(11, 9))
colors  = [SECONDARY if g>0 else DANGER for g in combined_sorted['gap']]
ax.barh(combined_sorted['country'], combined_sorted['gap'], color=colors, edgecolor='white', height=0.7)
ax.axvline(0, color='#333', lw=1.5)
ax.set_xlabel('Mobile − Broadband Speed (Mbps)', fontsize=11)
ax.set_title('Mobile vs Broadband Gap — Top & Bottom Countries (2025)', fontsize=13, fontweight='bold', color=PRIMARY)
blue_p = mpatches.Patch(color=SECONDARY, label='Mobile faster')
red_p  = mpatches.Patch(color=DANGER,    label='Broadband faster')
ax.legend(handles=[blue_p, red_p], fontsize=9)
plt.tight_layout()
plt.show()


## 9. Correlation Analysis
Does having fast broadband imply fast mobile? The correlation coefficient quantifies this relationship.


In [ ]:
corr = df_both['broadband'].corr(df_both['mobile'])
print(f'Pearson Correlation (Broadband vs Mobile): {corr:.4f}')
print(f'Interpretation: {"Moderate" if 0.3<abs(corr)<0.7 else "Strong" if abs(corr)>=0.7 else "Weak"} positive correlation')
print('\nThis means countries with faster broadband tend to also have faster mobile,')
print('but the relationship is not deterministic — infrastructure investment strategies differ significantly.')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Heatmap
corr_matrix = df_both[['broadband','mobile']].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='Blues', linewidths=0.5,
            linecolor='white', ax=axes[0], cbar_kws={'shrink':0.8},
            xticklabels=['Broadband','Mobile'], yticklabels=['Broadband','Mobile'])
axes[0].set_title('Correlation Matrix', fontsize=12, fontweight='bold', color=PRIMARY)

# Scatter with regression
axes[1].scatter(df_both['broadband'], df_both['mobile'], alpha=0.6, color=SECONDARY, edgecolors='white', s=50)
m, b = np.polyfit(df_both['broadband'], df_both['mobile'], 1)
x_line = np.linspace(df_both['broadband'].min(), df_both['broadband'].max(), 100)
axes[1].plot(x_line, m*x_line+b, color=DANGER, lw=2, label=f'Regression: y={m:.2f}x+{b:.1f}')
axes[1].set_xlabel('Broadband (Mbps)'); axes[1].set_ylabel('Mobile (Mbps)')
axes[1].set_title(f'Regression — r={corr:.3f}', fontsize=12, fontweight='bold', color=PRIMARY)
axes[1].legend(fontsize=9)

plt.suptitle('Broadband vs Mobile — Correlation Analysis', fontsize=13, fontweight='bold', color=PRIMARY)
plt.tight_layout()
plt.show()


## 10. Key Business Insights & Recommendations

---

### 🔑 Finding 1 — Extreme Global Disparity
The **digital divide is vast**: Singapore leads broadband at **410 Mbps** while Syria trails at just **3.5 Mbps** — a **116× difference**. For business analysts evaluating market entry, infrastructure quality is a critical risk dimension.

**→ Recommendation:** Businesses targeting digital-heavy services (SaaS, streaming, e-commerce) should prioritise Europe, East Asia, and North America where speeds reliably exceed 100 Mbps.

---

### 🔑 Finding 2 — Middle East Dominates Mobile
The UAE (**652 Mbps**), Qatar (**515 Mbps**), and Kuwait (**384 Mbps**) lead the world in **mobile speeds** — far outpacing their broadband — reflecting heavy 5G investment.

**→ Recommendation:** Mobile-first product strategies are essential for Middle East expansion. Optimise apps for mobile delivery rather than relying on broadband assumptions.

---

### 🔑 Finding 3 — Africa & South Asia Lag Significantly
Africa (avg broadband: **39.9 Mbps**) and South Asia (**43.2 Mbps**) are well below the global mean of **112 Mbps**. However, mobile coverage data is missing for 33+ African nations, limiting visibility.

**→ Recommendation:** For African/South Asian markets, design for **low-bandwidth environments** (lightweight apps, offline-capable features). Advocate for policy-level infrastructure investment.

---

### 🔑 Finding 4 — South America's Broadband Paradox
South America has the **highest average broadband** among all regions (**168.2 Mbps**) driven by Chile, Peru, and Brazil — yet mobile speeds lag (**62.1 Mbps avg**). This urban/rural and infrastructure split signals uneven development.

**→ Recommendation:** Dual-channel strategies in South America: broadband-optimised desktop products in urban centres, lightweight mobile apps for rural outreach.

---

### 🔑 Finding 5 — Moderate Broadband↔Mobile Correlation (r = 0.46)
The moderate correlation means a country's broadband quality does **not guarantee** mobile quality. Countries like Bulgaria and Bulgaria exploit mobile-first 5G without top broadband.

**→ Recommendation:** Always analyse both metrics independently when assessing a country's digital readiness. Composite "connectivity scores" should be dual-weighted.

---

### 📊 Data Quality Note
Mobile data is **missing for 53 countries (33.8%)**, skewing regional mobile comparisons. Any global ranking of mobile speed should be treated with caution until coverage improves.

---

*Analysis by Yash Gupta | Data: Speedtest Global Index 2025 | Tools: Python, Pandas, Matplotlib, Seaborn*
